# **Libraries**

In [1]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (Model, Environment, ManagedOnlineEndpoint, 
     ManagedOnlineDeployment, CodeConfiguration, DeploymentCollection, DataCollector)

from azure.identity import DefaultAzureCredential
import json
import os 
import requests

# **Get ML Client**

In [2]:
ml_client = MLClient.from_config(credential= DefaultAzureCredential())

Found the config file in: /config.json


In [3]:
ml_client

MLClient(credential=<azure.identity._credentials.default.DefaultAzureCredential object at 0x76489fdd3760>,
         subscription_id=cb199ddd-a44a-4bd8-9e03-3a31418f30d0,
         resource_group_name=rg-pure-prem,
         workspace_name=purprem_ws)

# **Register Model and Environment**

In [4]:
%cat env.yml

name: lgbm-env
channels: [conda-forge]
dependencies:
  - python=3.10
  - pip
  - pip:
      - lightgbm==4.5.0
      - scikit-learn==1.5
      - scipy==1.13
      - pandas==2.2
      - numpy==1.26
      - pydantic
      - azureml-inference-server-http
      - azureml-ai-monitoring

In [6]:
model = ml_client.models.create_or_update(
    Model(path = 'loss_cost_lgbm.txt', name = 'Loss_Cost_LGBM', type='custom_model')
)
env = ml_client.environments.create_or_update(
    Environment(
        name = 'Loss_Cost_LGBM_Env',
        conda_file = 'env.yml',
        image = 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04' 
    )
)

# **Register Endpoint**

In [7]:
endpoint = ManagedOnlineEndpoint(
    name = 'loss-cost-lgbm-endpoint-v4',
    auth_mode = 'key'
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.17.1) and child packages mlflow-skinny (3.15.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ManagedOnlineEndpoint({'public_network_access': <PublicNetworkAccessType.ENABLED: 'Enabled'>, 'provisioning_state': <EndpointProvisioningState.SUCCEEDED: 'Succeeded'>, 'scoring_uri': 'https://loss-cost-lgbm-endpoint-v4.canadacentral.inference.ml.azure.com/score', 'openapi_uri': 'https://loss-cost-lgbm-endpoint-v4.canadacentral.inference.ml.azure.com/swagger.json', 'name': 'loss-cost-lgbm-endpoint-v4', 'description': None, 'tags': {}, 'properties': {'createdBy': 'Mezmur Edo', 'createdAt': '2026-11-03T17:37:40.206259+0000', 'lastModifiedAt': '2026-11-03T18:43:36.742890+0000', 'azureml.onlineendpointid': '/subscriptions/cb199ddd-a44a-4bd8-9e03-3a31418f30d0/resourcegroups/rg-pure-prem/providers/microsoft.machinelearningservices/workspaces/purprem_ws/onlineendpoints/loss-cost-lgbm-endpoint-v4', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/cb199ddd-a44a-4bd8-9e03-3a31418f30d0/providers/Microsoft.MachineLearningServices/locations/canadacentral/mfeOperationsStatus/oeid

# **Deploy**

In [8]:
%cat predict.py

"""predict.py
"""
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import pandas as pd
import lightgbm as lgb
from functools import lru_cache
from features import build_features
from config import Config
import json
from azureml.ai.monitoring import Collector

def init():
    global model, inputs_collector, outputs_collector
    model = lgb.Booster(model_file = 'loss_cost_lgbm.txt')
    inputs_collector = Collector(name = 'model_inputs')
    outputs_collector = Collector(name = 'model_outputs')

def run(raw_data):
    policy_df = pd.DataFrame([json.loads(raw_data)])
    feature_df = build_features(policy_df)
    # collect engineered features
    context = inputs_collector.collect(feature_df)
    loss_cost = model.predict(feature_df)

    output =  {'loss_cost': float(loss_cost[0])}
    # collect output
    outputs_collector.collect(pd.DataFrame([output]), context)
    return output

# **Model Data Collector**

We will use Azure's built-in **Model Data Collector** to persist engineered features and predictions for monitoring and retraining jobs.

In [9]:
data_collector = DataCollector(
    collections = {
        'model_inputs' : DeploymentCollection(enabled = 'true'),
        'model_outputs': DeploymentCollection(enabled = 'true')
    },
    sampling_rate = 1.0
)

Class DeploymentCollection: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class DataCollector: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [10]:
deployment = ManagedOnlineDeployment(
    name = 'blue',
    endpoint_name= 'loss-cost-lgbm-endpoint-v4',
    model = model,
    environment = env,
    code_configuration = CodeConfiguration(code  = '.', scoring_script = 'predict.py'),
    instance_type = 'Standard_DS2_v2',
    instance_count = 1,
    data_collector = data_collector
)

ml_client.online_deployments.begin_create_or_update(deployment).result()

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint loss-cost-lgbm-endpoint-v4 exists
Uploading lgbm (0.09 MBs): 100%|██████████| 88479/88479 [00:00<00:00, 566717.00it/s]




....................................................................................................

ManagedOnlineDeployment({'private_network_connection': None, 'package_model': False, 'provisioning_state': <DeploymentProvisioningState.SUCCEEDED: 'Succeeded'>, 'endpoint_name': 'loss-cost-lgbm-endpoint-v4', 'type': 'Managed', 'name': 'blue', 'description': None, 'tags': {}, 'properties': {'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/cb199ddd-a44a-4bd8-9e03-3a31418f30d0/providers/Microsoft.MachineLearningServices/locations/canadacentral/mfeOperationsStatus/odidp:cf8a424c-550c-4760-8e2d-75cd6579165d:76b34d21-2b29-4364-a228-74b67d68d0a6?api-version=2023-04-01-preview'}, 'print_as_yaml': False, 'id': '/subscriptions/cb199ddd-a44a-4bd8-9e03-3a31418f30d0/resourceGroups/rg-pure-prem/providers/Microsoft.MachineLearningServices/workspaces/purprem_ws/onlineEndpoints/loss-cost-lgbm-endpoint-v4/deployments/blue', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/azuremlcompute/code/Users/mezmurw.edo/lgbm', 'creation_context': {'cre

# **Usage via Requests**

In [11]:
endpoint = ml_client.online_endpoints.get('loss-cost-lgbm-endpoint-v4')
scoring_uri = endpoint.scoring_uri
keys = ml_client.online_endpoints.get_keys('loss-cost-lgbm-endpoint-v4')
key = keys.primary_key

In [12]:
payload = {
    "RiskArea": 5.0, 
    "BonusMalus": 100, 
    "DrivAge": 35, 
    "LicAge": 120, 
    "VehUsage": "Private"
}

headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {key}',
    'azureml-model-deployment': 'blue'
}

response = requests.post(
    scoring_uri,
    headers = headers,
    json = payload,
)

result = response.json()
print(result)

{'loss_cost': 570.4891651846364}


In [13]:
ml_client.online_endpoints.begin_delete(name="loss-cost-lgbm-endpoint-v2").result()
print("Successfully deleted endpoint.")

ml_client.online_deployments.begin_delete(
    name='blue',
    endpoint_name='loss-cost-lgbm-endpoint-v4'
).result()
print("Successfully deleted deployment.")

Successfully deleted endpoint.
Successfully deleted deployment.
